# SA - DMA AXI STREAM vs AXI FULL


## SA Version DMA AXI STREAM

In [4]:
# ============================================================
# SA_df + Benchmark robuste (warmup/mesures + tableaux)
# ============================================================
from pynq import Overlay, allocate, PL
import numpy as np
import pandas as pd
import time
from IPython.display import display

# =========================
# CONFIG
# =========================
BITFILE   = "sadfsnr.bit"
SIG_CSV   = "signal.csv"
PRN_CSV   = "PRN-25.csv"

N         = 11999
NB_PHASES = 1023
FD_START  = -10000
FD_END    = 10000
FD_STEP   = 250
NB_FD     = ((FD_END - FD_START) // FD_STEP) + 1
TOTAL_OUT = NB_FD * NB_PHASES

# Niveau 2 (NCI)
M_NCI     = 1      # 5 / 10 / 20
K_CFAR    = 3.0

# Benchmark
WARMUP_RUNS  = 2
MEASURE_RUNS = 10
TIMEOUT_S    = 30.0
POLL_S       = 0.002

# AXI DMA regs
MM2S_DMACR = 0x00
MM2S_DMASR = 0x04
S2MM_DMACR = 0x30
S2MM_DMASR = 0x34
ERR_MASK   = (1 << 4) | (1 << 5) | (1 << 6) | (1 << 8) | (1 << 9) | (1 << 10) | (1 << 14)

# AXI-Lite regs stables
ACQ_CTRL    = 0x00
ACQ_FDSTEP  = 0x40


def decode_dmasr(v):
    return {
        "halted": bool(v & (1 << 0)),
        "idle": bool(v & (1 << 1)),
        "dma_int_err": bool(v & (1 << 4)),
        "dma_slv_err": bool(v & (1 << 5)),
        "dma_dec_err": bool(v & (1 << 6)),
        "ioc_irq": bool(v & (1 << 12)),
        "err_irq": bool(v & (1 << 14)),
    }


def dma_hard_reset_chan(dma, cr_off, sr_off, run_after=True):
    ctrl = dma.mmio.read(cr_off)
    dma.mmio.write(cr_off, ctrl | 0x4)
    t0 = time.time()
    while time.time() - t0 < 2.0:
        if (dma.mmio.read(cr_off) & 0x4) == 0:
            break
        time.sleep(0.001)
    if run_after:
        dma.mmio.write(cr_off, 0x00010001)
    return dma.mmio.read(sr_off)


def dma_full_reset(dma, has_mm2s=True, has_s2mm=True):
    if has_mm2s:
        dma_hard_reset_chan(dma, MM2S_DMACR, MM2S_DMASR, run_after=True)
    if has_s2mm:
        dma_hard_reset_chan(dma, S2MM_DMACR, S2MM_DMASR, run_after=True)


def dmas_no_error(dma_sig, dma_prn, dma_out):
    s_sig = dma_sig.mmio.read(MM2S_DMASR)
    s_prn = dma_prn.mmio.read(MM2S_DMASR)
    s_out = dma_out.mmio.read(S2MM_DMASR)
    return ((s_sig & ERR_MASK) == 0) and ((s_prn & ERR_MASK) == 0) and ((s_out & ERR_MASK) == 0)


def rm_read_int(rm, name_candidates, default=None):
    for n in name_candidates:
        if hasattr(rm, n):
            obj = getattr(rm, n)
            try:
                return int(obj)
            except Exception:
                try:
                    return int(obj.read())
                except Exception:
                    pass
    return default


def ip_read_outputs(acq):
    rm = acq.register_map
    doppler = rm_read_int(rm, ["doppler_out", "doppler"])
    codephase = rm_read_int(rm, ["codephase_out", "codephase"])
    sat = rm_read_int(rm, ["sat_detected", "sat"])
    rx_count = rm_read_int(rm, ["rx_count", "rx_count_out"])
    prn_count = rm_read_int(rm, ["prn_count", "prn_count_out"])
    max_pwr = rm_read_int(rm, ["max_power_out", "max_power"])
    mean_pwr = rm_read_int(rm, ["mean_power_out", "mean_power"])

    if doppler is not None and doppler > 0x7FFFFFFF:
        doppler -= 0x100000000

    return {
        "doppler": doppler,
        "codephase": codephase,
        "sat": sat,
        "rx_count": rx_count,
        "prn_count": prn_count,
        "max_power_out": max_pwr,
        "mean_power_out": mean_pwr,
    }


def run_one_acq(acq, dma_sig, dma_prn, dma_out, in_sig, in_prn, out_buf, fd_step=FD_STEP, timeout_s=TIMEOUT_S):
    out_buf[:] = 0
    out_buf.flush()

    acq.write(ACQ_FDSTEP, int(fd_step))

    dma_out.recvchannel.transfer(out_buf)
    acq.write(ACQ_CTRL, 0x01)
    dma_sig.sendchannel.transfer(in_sig)
    dma_prn.sendchannel.transfer(in_prn)

    t0 = time.time()
    state = "RUNNING"
    while True:
        if not dmas_no_error(dma_sig, dma_prn, dma_out):
            state = "DMA_ERROR"
            break
        ctrl = acq.read(ACQ_CTRL)
        if ctrl & 0x2:
            state = "AP_DONE"
            break
        if (time.time() - t0) > timeout_s:
            state = "TIMEOUT"
            break
        time.sleep(POLL_S)

    try:
        dma_sig.sendchannel.wait()
        dma_prn.sendchannel.wait()
        dma_out.recvchannel.wait()
    except Exception:
        state = "DMA_WAIT_EXC"

    out_buf.invalidate()
    regs = ip_read_outputs(acq)

    return {
        "state": state,
        "regs": regs,
        "corr": np.array(out_buf, dtype=np.int32, copy=True),
    }


def _style_table(df, title, float_cols, n_col="N"):
    fmt = {c: "{:.6f}" for c in float_cols}
    if n_col in df.columns:
        fmt[n_col] = lambda x: f"{int(x)}"

    styled = (
        df.style
        .hide_index()
        .set_caption(title)
        .format(fmt)
        .set_table_styles([
            {"selector": "caption", "props": [
                ("caption-side", "top"),
                ("font-weight", "bold"),
                ("font-size", "14px"),
                ("text-align", "left"),
                ("padding", "8px 0")
            ]},
            {"selector": "th", "props": [
                ("background-color", "#f2f2f2"),
                ("text-align", "center"),
                ("border", "1px solid #cccccc"),
                ("padding", "6px")
            ]},
            {"selector": "td", "props": [
                ("text-align", "center"),
                ("border", "1px solid #cccccc"),
                ("padding", "6px")
            ]},
            {"selector": "table", "props": [
                ("border-collapse", "collapse"),
                ("width", "100%")
            ]},
        ])
    )
    display(styled)


def _build_stats_df(measures_dict):
    rows = []
    for name, values in measures_dict.items():
        s = pd.Series(values, dtype="float64")
        rows.append({
            "Mesure": name,
            "N": int(s.size),
            "Mean": float(s.mean()),
            "Std": float(s.std(ddof=0)),
            "Median": float(s.median()),
            "Min": float(s.min()),
            "Max": float(s.max()),
            "P95": float(s.quantile(0.95)),
        })
    return pd.DataFrame(rows)


# ============================================================
# EXEC
# ============================================================
in_sig = in_prn = out_buf = None

try:
    try:
        PL.reset()
    except Exception:
        pass

    ol = Overlay(BITFILE, download=True)
    time.sleep(1.0)

    acq = ol.acquisition_serial_0
    dma_sig = ol.axi_dma_0
    dma_prn = ol.axi_dma_1
    dma_out = ol.axi_dma_2

#     print("IP register_map:")
#     print(acq.register_map)

    # Reset DMA initial
    dma_full_reset(dma_sig, has_mm2s=True, has_s2mm=False)
    dma_full_reset(dma_prn, has_mm2s=True, has_s2mm=False)
    dma_full_reset(dma_out, has_mm2s=False, has_s2mm=True)
    time.sleep(0.1)

    # Charger données
    sig_raw = np.genfromtxt(SIG_CSV, delimiter=",").astype(np.int32).ravel()
    prn_raw = np.genfromtxt(PRN_CSV, delimiter=",").astype(np.int32).ravel()

    if len(sig_raw) < N:
        raise RuntimeError(f"Signal CSV trop court: {len(sig_raw)} < {N}")
    if len(prn_raw) < N:
        raise RuntimeError(f"PRN CSV trop court: {len(prn_raw)} < {N}")

    in_sig = allocate(shape=(N,), dtype=np.int32)
    in_prn = allocate(shape=(N,), dtype=np.int32)
    out_buf = allocate(shape=(TOTAL_OUT,), dtype=np.int32)

    in_prn[:] = prn_raw[:N]
    in_prn.flush()

    bytes_in_per_acq = in_sig.nbytes + in_prn.nbytes
    bytes_out_per_acq = out_buf.nbytes
    print(f"[INFO] DMA bytes_in_per_acq={bytes_in_per_acq} ({bytes_in_per_acq//1024} KB) | "
          f"bytes_out_per_acq={bytes_out_per_acq} ({bytes_out_per_acq//1024} KB)")

    records = []
    total_runs = WARMUP_RUNS + MEASURE_RUNS
    print(f"[INFO] Benchmark: {WARMUP_RUNS} warmup + {MEASURE_RUNS} mesures")

    for r in range(total_runs):
        warmup = (r < WARMUP_RUNS)

        # 1) cache/prepare
        t0 = time.perf_counter_ns()
        in_prn.flush()
        out_buf.flush()
        t1 = time.perf_counter_ns()

        # 2) config
        acq.write(ACQ_FDSTEP, int(FD_STEP))
        t2 = time.perf_counter_ns()

        # 3) run NCI
        accum = np.zeros(TOTAL_OUT, dtype=np.float64)
        ok_runs = 0
        all_states = []
        last_regs = None
        ip_wait_ns = 0

        for m in range(M_NCI):
            s = m * N
            e = s + N
            if e <= len(sig_raw):
                in_sig[:] = sig_raw[s:e]
            else:
                in_sig[:] = sig_raw[:N]
            in_sig.flush()

            t_run0 = time.perf_counter_ns()
            res = run_one_acq(acq, dma_sig, dma_prn, dma_out, in_sig, in_prn, out_buf)
            t_run1 = time.perf_counter_ns()
            ip_wait_ns += (t_run1 - t_run0)

            all_states.append(res["state"])
            last_regs = res["regs"]

            if res["state"] != "AP_DONE":
                continue

            accum += res["corr"].astype(np.float64)
            ok_runs += 1

        if ok_runs == 0:
            cache_ms = (t1 - t0) / 1e6
            cfg_ms = (t2 - t1) / 1e6
            ip_ms = ip_wait_ns / 1e6
            post_ms = 0.0
            total_ms = cache_ms + cfg_ms + ip_ms

            records.append({
                "run": r,
                "warmup": warmup,
                "valid": False,
                "state": "NO_AP_DONE",
                "cache_ms": cache_ms,
                "cfg_ms": cfg_ms,
                "ip_ms": ip_ms,
                "post_ms": post_ms,
                "total_ms": total_ms,
                "bw_in_total_MBps": np.nan,
                "bw_out_total_MBps": np.nan,
                "bw_in_ip_MBps": np.nan,
                "bw_out_ip_MBps": np.nan,
                "doppler": np.nan,
                "codephase": np.nan,
                "sat": np.nan,
                "corr_sig": np.nan,
                "corr_max": np.nan,
                "corr_max_idx": np.nan,
                "ok_runs": ok_runs,
                "states": str(all_states),
            })
            continue

        # 4) post-traitement / décision NCI
        t4 = time.perf_counter_ns()
        max_idx = int(np.argmax(accum))
        max_val = float(accum[max_idx])
        mean_val = float(np.mean(accum))
        threshold = mean_val * (1.0 + K_CFAR / np.sqrt(ok_runs))
        detected = int(max_val > threshold)

        fd_idx = max_idx // NB_PHASES
        tau_idx = max_idx % NB_PHASES
        doppler_nci = FD_START + fd_idx * FD_STEP
        codephase_nci = tau_idx

        corr_arr = accum.astype(np.int64, copy=False)
        corr_sig = int(corr_arr[0] + corr_arr[TOTAL_OUT // 2] + corr_arr[-1])
        corr_max = int(max_val)
        corr_max_idx = int(max_idx)
        t5 = time.perf_counter_ns()

        cache_ms = (t1 - t0) / 1e6
        cfg_ms = (t2 - t1) / 1e6
        ip_ms = ip_wait_ns / 1e6
        post_ms = (t5 - t4) / 1e6
        total_ms = (t5 - t0) / 1e6

        bytes_in = bytes_in_per_acq * ok_runs
        bytes_out = bytes_out_per_acq * ok_runs

        bw_in_total_MBps = (bytes_in / (total_ms / 1000.0)) / 1e6 if total_ms > 0 else np.nan
        bw_out_total_MBps = (bytes_out / (total_ms / 1000.0)) / 1e6 if total_ms > 0 else np.nan
        bw_in_ip_MBps = (bytes_in / (ip_ms / 1000.0)) / 1e6 if ip_ms > 0 else np.nan
        bw_out_ip_MBps = (bytes_out / (ip_ms / 1000.0)) / 1e6 if ip_ms > 0 else np.nan

        records.append({
            "run": r,
            "warmup": warmup,
            "valid": True,
            "state": "AP_DONE",
            "cache_ms": cache_ms,
            "cfg_ms": cfg_ms,
            "ip_ms": ip_ms,
            "post_ms": post_ms,
            "total_ms": total_ms,
            "bw_in_total_MBps": bw_in_total_MBps,
            "bw_out_total_MBps": bw_out_total_MBps,
            "bw_in_ip_MBps": bw_in_ip_MBps,
            "bw_out_ip_MBps": bw_out_ip_MBps,
            "doppler": int(doppler_nci),
            "codephase": int(codephase_nci),
            "sat": int(detected),
            "corr_sig": int(corr_sig),
            "corr_max": int(corr_max),
            "corr_max_idx": int(corr_max_idx),
            "ok_runs": int(ok_runs),
            "states": str(all_states),
            "max_power_out": last_regs.get("max_power_out") if last_regs else None,
            "mean_power_out": last_regs.get("mean_power_out") if last_regs else None,
        })

    # ============================================================
    # TABLEAUX
    # ============================================================
    df_all = pd.DataFrame(records)
    meas = df_all[(~df_all["warmup"]) & (df_all["valid"])].copy()

    if len(meas) == 0:
        raise RuntimeError("Aucune mesure valide hors warmup.")

    table_a = _build_stats_df({
        "FPGA prep cache": [x / 1000.0 for x in meas["cache_ms"].tolist()],
        "FPGA config registres": [x / 1000.0 for x in meas["cfg_ms"].tolist()],
        "FPGA IP calcul": [x / 1000.0 for x in meas["ip_ms"].tolist()],
        "FPGA post-traitement": [x / 1000.0 for x in meas["post_ms"].tolist()],
        "FPGA end-to-end": [x / 1000.0 for x in meas["total_ms"].tolist()],
    }).rename(columns={
        "Mean": "Mean (s)", "Std": "Std (s)", "Median": "Median (s)",
        "Min": "Min (s)", "Max": "Max (s)", "P95": "P95 (s)",
    })
    _style_table(table_a, "TABLEAU A - PERFORMANCE SA_df (DMA)",
                 ["Mean (s)", "Std (s)", "Median (s)", "Min (s)", "Max (s)", "P95 (s)"])

    table_b = _build_stats_df({
        "Debit IN total": meas["bw_in_total_MBps"].tolist(),
        "Debit OUT total": meas["bw_out_total_MBps"].tolist(),
        "Debit IN IP": meas["bw_in_ip_MBps"].tolist(),
        "Debit OUT IP": meas["bw_out_ip_MBps"].tolist(),
    }).rename(columns={
        "Mean": "Mean (MB/s)", "Std": "Std (MB/s)", "Median": "Median (MB/s)",
        "Min": "Min (MB/s)", "Max": "Max (MB/s)", "P95": "P95 (MB/s)",
    })
    _style_table(table_b, "TABLEAU B - DEBITS SA_df (DMA)",
                 ["Mean (MB/s)", "Std (MB/s)", "Median (MB/s)", "Min (MB/s)", "Max (MB/s)", "P95 (MB/s)"])

    dop_set = sorted(set(int(v) for v in meas["doppler"].tolist()))
    cp_set  = sorted(set(int(v) for v in meas["codephase"].tolist()))
    sat_set = sorted(set(int(v) for v in meas["sat"].tolist()))
    sig_set = sorted(set(int(v) for v in meas["corr_sig"].tolist()))

    table_c = pd.DataFrame([
        {"Signal": "doppler",   "Stable": "Oui" if len(dop_set) == 1 else "Non", "Valeurs uniques": str(dop_set[:8])},
        {"Signal": "codephase", "Stable": "Oui" if len(cp_set)  == 1 else "Non", "Valeurs uniques": str(cp_set[:8])},
        {"Signal": "sat",       "Stable": "Oui" if len(sat_set) == 1 else "Non", "Valeurs uniques": str(sat_set[:8])},
        {"Signal": "corr_sig",  "Stable": "Oui" if len(sig_set) == 1 else "Non", "Valeurs uniques": str(sig_set[:8])},
    ])
    _style_table(table_c, "TABLEAU C - STABILITE INTER-RUNS (DMA)", [])

    table_d = pd.DataFrame([
        {
            "Run": int(x["run"]),
            "corr_max": int(x["corr_max"]),
            "corr_max_idx": int(x["corr_max_idx"]),
            "fd_correspondant (Hz)": FD_START + (int(x["corr_max_idx"]) // NB_PHASES) * FD_STEP,
            "phase_correspondante": int(x["corr_max_idx"]) % NB_PHASES,
        }
        for _, x in meas.iterrows()
    ])
    _style_table(table_d, "TABLEAU D - MAX CORRELATION PAR RUN (DMA)", [])

    if "BENCH_SNAPSHOTS" not in globals():
        BENCH_SNAPSHOTS = {}
    BENCH_SNAPSHOTS["DMA AXI STREAM"] = df_all.copy()
    print("[OK] Snapshot auto enregistre: DMA AXI STREAM")
    print(f"[OK] Runs valides hors warmup: {len(meas)} / {MEASURE_RUNS}")

finally:
    if in_sig is not None:
        in_sig.freebuffer()
    if in_prn is not None:
        in_prn.freebuffer()
    if out_buf is not None:
        out_buf.freebuffer()

[INFO] DMA bytes_in_per_acq=95992 (93 KB) | bytes_out_per_acq=331452 (323 KB)
[INFO] Benchmark: 2 warmup + 10 mesures


Mesure,N,Mean (s),Std (s),Median (s),Min (s),Max (s),P95 (s)
FPGA prep cache,10,0.000474,0.000009,0.000477,0.000460,0.000484,0.000483
FPGA config registres,10,0.000092,0.000008,0.000090,0.000085,0.000113,0.000104
FPGA IP calcul,10,2.015409,0.000528,2.015252,2.014480,2.016148,2.016129
FPGA post-traitement,10,0.012417,0.000276,0.012487,0.011985,0.012921,0.012792
FPGA end-to-end,10,2.038949,0.000364,2.038866,2.038458,2.039902,2.039541


Mesure,N,Mean (MB/s),Std (MB/s),Median (MB/s),Min (MB/s),Max (MB/s),P95 (MB/s)
Debit IN total,10,0.047079,0.000008,0.047081,0.047057,0.047090,0.047088
Debit OUT total,10,0.162560,0.000029,0.162567,0.162484,0.162599,0.162591
Debit IN IP,10,0.047629,0.000012,0.047633,0.047612,0.047651,0.047646
Debit OUT IP,10,0.164459,0.000043,0.164472,0.164399,0.164535,0.164519


Signal,Stable,Valeurs uniques
doppler,Oui,[-1750]
codephase,Oui,[150]
sat,Oui,[1]
corr_sig,Oui,[30029]


Run,corr_max,corr_max_idx,fd_correspondant (Hz),phase_correspondante
2,1935592,33909,-1750,150
3,1935592,33909,-1750,150
4,1935592,33909,-1750,150
5,1935592,33909,-1750,150
6,1935592,33909,-1750,150
7,1935592,33909,-1750,150
8,1935592,33909,-1750,150
9,1935592,33909,-1750,150
10,1935592,33909,-1750,150
11,1935592,33909,-1750,150


[OK] Snapshot auto enregistre: DMA AXI STREAM
[OK] Runs valides hors warmup: 10 / 10


## SA Version AXI FULL

In [5]:
# ===== CONFIG =====
BITFILE = "SAdf_m_axi.bit"
SIG_CSV = "signal.csv"
PRN_CSV = "PRN-25.csv"
IP_NAME = "acquisition_serial_m_0"

N = 11999
NB_PHASES = 1023
FD_START = -10000
FD_END = 10000
FD_STEP = 250

WARMUP_RUNS  = 2
MEASURE_RUNS = 10
TIMEOUT_S    = 60.0

import time
import numpy as np
import pandas as pd
from pynq import Overlay, allocate
from IPython.display import display

REG = {
    "AP_CTRL":             0x00,
    "RX_REAL_DATA":        0x10,
    "PRN_IN_DATA":         0x1C,
    "CORR_OUT_DATA":       0x28,
    "CORR_COUNT_DATA":     0x34,
    "CORR_COUNT_CTRL":     0x38,
    "DOPPLER_OUT_DATA":    0x44,
    "DOPPLER_OUT_CTRL":    0x48,
    "CODEPHASE_OUT_DATA":  0x54,
    "CODEPHASE_OUT_CTRL":  0x58,
    "SAT_DETECTED_DATA":   0x64,
    "SAT_DETECTED_CTRL":   0x68,
    "FD_STEP_DATA":        0x74,
    "MAX_POWER_OUT_DATA":  0x7C,
    "MAX_POWER_OUT_CTRL":  0x80,
    "MEAN_POWER_OUT_DATA": 0x8C,
    "MEAN_POWER_OUT_CTRL": 0x90,
    "RX_COUNT_DATA":       0x9C,
    "PRN_COUNT_DATA":      0xAC,
    "RX_LAST_SEEN_DATA":   0xBC,
    "PRN_LAST_SEEN_DATA":  0xCC,
    "RX_LAST_POS_DATA":    0xDC,
    "PRN_LAST_POS_DATA":   0xEC,
}

def load_csv_int(path):
    data = np.genfromtxt(path, delimiter=",", dtype=np.int32)
    if np.isscalar(data):
        data = np.array([int(data)], dtype=np.int32)
    elif data.ndim > 1:
        data = data.ravel().astype(np.int32)
    return data.astype(np.int32)

def write_u64(ip, base_offset, value):
    v = int(value) & 0xFFFFFFFFFFFFFFFF
    lo = v & 0xFFFFFFFF
    hi = (v >> 32) & 0xFFFFFFFF
    ip.write(base_offset, lo)
    ip.write(base_offset + 4, hi)

def read_u32(ip, offset):
    return int(ip.read(offset)) & 0xFFFFFFFF

def to_int32(x):
    x &= 0xFFFFFFFF
    return x if x < 0x80000000 else x - 0x100000000

def _style_table(df, title, float_cols, n_col="N"):
    fmt = {c: "{:.6f}" for c in float_cols}
    if n_col in df.columns:
        fmt[n_col] = lambda x: f"{int(x)}"
    styled = (
        df.style
        .hide_index()
        .set_caption(title)
        .format(fmt)
        .set_table_styles([
            {"selector": "caption", "props": [
                ("caption-side", "top"), ("font-weight", "bold"),
                ("font-size", "14px"), ("text-align", "left"), ("padding", "8px 0")
            ]},
            {"selector": "th", "props": [
                ("background-color", "#f2f2f2"), ("text-align", "center"),
                ("border", "1px solid #cccccc"), ("padding", "6px")
            ]},
            {"selector": "td", "props": [
                ("text-align", "center"), ("border", "1px solid #cccccc"), ("padding", "6px")
            ]},
            {"selector": "table", "props": [
                ("border-collapse", "collapse"), ("width", "100%")
            ]},
        ])
    )
    display(styled)

def _build_stats_df(measures_dict):
    rows = []
    for name, values in measures_dict.items():
        s = pd.Series(values, dtype="float64")
        rows.append({
            "Mesure": name,
            "N": int(s.size),
            "Mean": float(s.mean()),
            "Std": float(s.std(ddof=0)),
            "Median": float(s.median()),
            "Min": float(s.min()),
            "Max": float(s.max()),
            "P95": float(s.quantile(0.95)),
        })
    return pd.DataFrame(rows)

print("[INFO] Chargement overlay...")
ol = Overlay(BITFILE)
ip = getattr(ol, IP_NAME)
print(f"[OK] IP '{IP_NAME}' loaded")

print("[INFO] Chargement CSV...")
signal = load_csv_int(SIG_CSV)
prn = load_csv_int(PRN_CSV)

if signal.size < N or prn.size < N:
    raise ValueError(f"Taille insuffisante: signal={signal.size}, prn={prn.size}, N={N}")

signal = signal[:N]
prn = prn[:N]

nb_fd = (FD_END - FD_START) // FD_STEP + 1
total_outputs = nb_fd * NB_PHASES

rx_buf = allocate(shape=(N,), dtype=np.int32)
prn_buf = allocate(shape=(N,), dtype=np.int32)
corr_buf = allocate(shape=(total_outputs,), dtype=np.int32)

rx_buf[:] = signal
prn_buf[:] = prn
corr_buf[:] = 0

bytes_in = rx_buf.nbytes + prn_buf.nbytes
bytes_out = corr_buf.nbytes

print(f"[OK] Buffers: N={N}, nb_fd={nb_fd}, total_outputs={total_outputs}")
print(f"[INFO] AXI FULL bytes_in={bytes_in} ({bytes_in//1024} KB) | bytes_out={bytes_out} ({bytes_out//1024} KB)")

records = []
total_runs = WARMUP_RUNS + MEASURE_RUNS
print(f"[INFO] Benchmark: {WARMUP_RUNS} warmup + {MEASURE_RUNS} mesures")

for r in range(total_runs):
    warmup = (r < WARMUP_RUNS)
    corr_buf[:] = 0

    t0 = time.perf_counter_ns()
    rx_buf.flush()
    prn_buf.flush()
    corr_buf.flush()
    t1 = time.perf_counter_ns()

    write_u64(ip, REG["RX_REAL_DATA"], rx_buf.physical_address)
    write_u64(ip, REG["PRN_IN_DATA"], prn_buf.physical_address)
    write_u64(ip, REG["CORR_OUT_DATA"], corr_buf.physical_address)
    ip.write(REG["FD_STEP_DATA"], int(FD_STEP) & 0xFFFFFFFF)
    t2 = time.perf_counter_ns()

    ip.write(REG["AP_CTRL"], 0x01)
    t3 = time.perf_counter_ns()

    t_wait = time.perf_counter()
    while True:
        status = read_u32(ip, REG["AP_CTRL"])
        ap_done = (status >> 1) & 0x1
        if ap_done:
            break
        if (time.perf_counter() - t_wait) > TIMEOUT_S:
            raise TimeoutError(f"Timeout waiting for ap_done > {TIMEOUT_S}s")
        time.sleep(0.001)
    t4 = time.perf_counter_ns()

    corr_buf.invalidate()

    corr_count = read_u32(ip, REG["CORR_COUNT_DATA"])
    if corr_count == 0:
        raise RuntimeError("corr_count=0")
    if corr_count > total_outputs:
        raise RuntimeError(f"corr_count={corr_count} > corr_buf_size={total_outputs}")

    corr_valid = corr_buf[:corr_count]
    corr_max = int(np.max(corr_valid))
    corr_max_idx = int(np.argmax(corr_valid))
    doppler = to_int32(read_u32(ip, REG["DOPPLER_OUT_DATA"]))
    doppler_vld = read_u32(ip, REG["DOPPLER_OUT_CTRL"]) & 0x1
    codephase = to_int32(read_u32(ip, REG["CODEPHASE_OUT_DATA"]))
    codephase_vld = read_u32(ip, REG["CODEPHASE_OUT_CTRL"]) & 0x1
    sat = to_int32(read_u32(ip, REG["SAT_DETECTED_DATA"]))
    sat_vld = read_u32(ip, REG["SAT_DETECTED_CTRL"]) & 0x1

    max_power_out = read_u32(ip, REG["MAX_POWER_OUT_DATA"])
    mean_power_out = read_u32(ip, REG["MEAN_POWER_OUT_DATA"])
    rx_count = read_u32(ip, REG["RX_COUNT_DATA"])
    prn_count = read_u32(ip, REG["PRN_COUNT_DATA"])
    rx_last_seen = to_int32(read_u32(ip, REG["RX_LAST_SEEN_DATA"]))
    prn_last_seen = to_int32(read_u32(ip, REG["PRN_LAST_SEEN_DATA"]))
    rx_last_pos = to_int32(read_u32(ip, REG["RX_LAST_POS_DATA"]))
    prn_last_pos = to_int32(read_u32(ip, REG["PRN_LAST_POS_DATA"]))

    corr_sig = int(np.int64(corr_valid[0]) + np.int64(corr_valid[corr_count // 2]) + np.int64(corr_valid[-1]))
    t5 = time.perf_counter_ns()

    cache_ms = (t1 - t0) / 1e6
    cfg_ms = (t2 - t1) / 1e6
    ip_ms = (t4 - t3) / 1e6
    post_ms = (t5 - t4) / 1e6
    total_ms = (t5 - t0) / 1e6

    bw_in_total_MBps = (bytes_in / (total_ms / 1000.0)) / 1e6
    bw_out_total_MBps = (bytes_out / (total_ms / 1000.0)) / 1e6
    bw_in_ip_MBps = (bytes_in / (ip_ms / 1000.0)) / 1e6 if ip_ms > 0 else np.nan
    bw_out_ip_MBps = (bytes_out / (ip_ms / 1000.0)) / 1e6 if ip_ms > 0 else np.nan

    records.append({
        "run": r,
        "warmup": warmup,
        "cache_ms": cache_ms,
        "cfg_ms": cfg_ms,
        "ip_ms": ip_ms,
        "post_ms": post_ms,
        "total_ms": total_ms,
        "bw_in_total_MBps": bw_in_total_MBps,
        "bw_out_total_MBps": bw_out_total_MBps,
        "bw_in_ip_MBps": bw_in_ip_MBps,
        "bw_out_ip_MBps": bw_out_ip_MBps,
        "doppler": doppler,
        "doppler_vld": doppler_vld,
        "codephase": codephase,
        "codephase_vld": codephase_vld,
        "sat": sat,
        "sat_vld": sat_vld,
        "corr_count": int(corr_count),
        "corr_sig": corr_sig,
        "corr_max": corr_max,
        "corr_max_idx": corr_max_idx,
        "max_power_out": int(max_power_out),
        "mean_power_out": int(mean_power_out),
        "rx_count": int(rx_count),
        "prn_count": int(prn_count),
        "rx_last_seen": int(rx_last_seen),
        "prn_last_seen": int(prn_last_seen),
        "rx_last_pos": int(rx_last_pos),
        "prn_last_pos": int(prn_last_pos),
        "ok_runs": 1,
    })

df_all = pd.DataFrame(records)
meas = [x for x in records if not x["warmup"]]
if len(meas) == 0:
    raise RuntimeError("Aucune mesure hors warmup. Mets MEASURE_RUNS >= 1.")

table_a = _build_stats_df({
    "FPGA prep cache": [x["cache_ms"] / 1000.0 for x in meas],
    "FPGA config registres": [x["cfg_ms"] / 1000.0 for x in meas],
    "FPGA IP calcul": [x["ip_ms"] / 1000.0 for x in meas],
    "FPGA post-traitement": [x["post_ms"] / 1000.0 for x in meas],
    "FPGA end-to-end": [x["total_ms"] / 1000.0 for x in meas],
}).rename(columns={
    "Mean": "Mean (s)", "Std": "Std (s)", "Median": "Median (s)",
    "Min": "Min (s)", "Max": "Max (s)", "P95": "P95 (s)",
})
_style_table(table_a, "TABLEAU A - PERFORMANCE AXI FULL",
             ["Mean (s)", "Std (s)", "Median (s)", "Min (s)", "Max (s)", "P95 (s)"])

table_b = _build_stats_df({
    "Debit IN total": [x["bw_in_total_MBps"] for x in meas],
    "Debit OUT total": [x["bw_out_total_MBps"] for x in meas],
    "Debit IN IP": [x["bw_in_ip_MBps"] for x in meas],
    "Debit OUT IP": [x["bw_out_ip_MBps"] for x in meas],
}).rename(columns={
    "Mean": "Mean (MB/s)", "Std": "Std (MB/s)", "Median": "Median (MB/s)",
    "Min": "Min (MB/s)", "Max": "Max (MB/s)", "P95": "P95 (MB/s)",
})
_style_table(table_b, "TABLEAU B - DEBITS AXI FULL",
             ["Mean (MB/s)", "Std (MB/s)", "Median (MB/s)", "Min (MB/s)", "Max (MB/s)", "P95 (MB/s)"])

dop_set = sorted(set(int(x["doppler"]) for x in meas))
cp_set = sorted(set(int(x["codephase"]) for x in meas))
sat_set = sorted(set(int(x["sat"]) for x in meas))
sig_set = sorted(set(int(x["corr_sig"]) for x in meas))

table_c = pd.DataFrame([
    {"Signal": "doppler", "Stable": "Oui" if len(dop_set) == 1 else "Non", "Valeurs uniques": str(dop_set[:8])},
    {"Signal": "codephase", "Stable": "Oui" if len(cp_set) == 1 else "Non", "Valeurs uniques": str(cp_set[:8])},
    {"Signal": "sat", "Stable": "Oui" if len(sat_set) == 1 else "Non", "Valeurs uniques": str(sat_set[:8])},
    {"Signal": "corr_sig", "Stable": "Oui" if len(sig_set) == 1 else "Non", "Valeurs uniques": str(sig_set[:8])},
])
_style_table(table_c, "TABLEAU C - STABILITE INTER-RUNS (AXI FULL)", [])

table_d = pd.DataFrame([
    {
        "Run": int(x["run"]),
        "corr_max": int(x["corr_max"]),
        "corr_max_idx": int(x["corr_max_idx"]),
        "fd_correspondant (Hz)": FD_START + (int(x["corr_max_idx"]) // NB_PHASES) * FD_STEP,
        "phase_correspondante": int(x["corr_max_idx"]) % NB_PHASES,
    }
    for x in meas
])
_style_table(table_d, "TABLEAU D - MAX CORRELATION PAR RUN (AXI FULL)", [])

if "BENCH_SNAPSHOTS" not in globals():
    BENCH_SNAPSHOTS = {}
BENCH_SNAPSHOTS["AXI FULL"] = df_all.copy()
print("[OK] Snapshot auto enregistre: AXI FULL")

print("[CHECK] Exemple attendu:")
print("codephase_out=150, max_power_out=1935592, sat=1, corr_count=82863, doppler=-1750")

rx_buf.freebuffer()
prn_buf.freebuffer()
corr_buf.freebuffer()
print("[OK] Benchmark termine.")

[INFO] Chargement overlay...
[OK] IP 'acquisition_serial_m_0' loaded
[INFO] Chargement CSV...
[OK] Buffers: N=11999, nb_fd=81, total_outputs=82863
[INFO] AXI FULL bytes_in=95992 (93 KB) | bytes_out=331452 (323 KB)
[INFO] Benchmark: 2 warmup + 10 mesures


Mesure,N,Mean (s),Std (s),Median (s),Min (s),Max (s),P95 (s)
FPGA prep cache,10,0.000506,0.000013,0.000511,0.000487,0.000518,0.000518
FPGA config registres,10,0.000260,0.000006,0.000261,0.000249,0.000266,0.000266
FPGA IP calcul,10,2.010612,0.000408,2.010492,2.010026,2.011138,2.011137
FPGA post-traitement,10,0.017511,0.000046,0.017518,0.017455,0.017611,0.017581
FPGA end-to-end,10,2.028922,0.000410,2.028858,2.028290,2.029490,2.029476


Mesure,N,Mean (MB/s),Std (MB/s),Median (MB/s),Min (MB/s),Max (MB/s),P95 (MB/s)
Debit IN total,10,0.047312,0.000010,0.047313,0.047299,0.047327,0.047326
Debit OUT total,10,0.163364,0.000033,0.163369,0.163318,0.163415,0.163414
Debit IN IP,10,0.047743,0.000010,0.047746,0.047730,0.047757,0.047756
Debit OUT IP,10,0.164851,0.000033,0.164861,0.164808,0.164899,0.164898


Signal,Stable,Valeurs uniques
doppler,Oui,[-1750]
codephase,Oui,[150]
sat,Oui,[1]
corr_sig,Oui,[30029]


Run,corr_max,corr_max_idx,fd_correspondant (Hz),phase_correspondante
2,1935592,33909,-1750,150
3,1935592,33909,-1750,150
4,1935592,33909,-1750,150
5,1935592,33909,-1750,150
6,1935592,33909,-1750,150
7,1935592,33909,-1750,150
8,1935592,33909,-1750,150
9,1935592,33909,-1750,150
10,1935592,33909,-1750,150
11,1935592,33909,-1750,150


[OK] Snapshot auto enregistre: AXI FULL
[CHECK] Exemple attendu:
codephase_out=150, max_power_out=1935592, sat=1, corr_count=82863, doppler=-1750
[OK] Benchmark termine.


## Comparaison complete DMA AXI STREAM vs AXI FULL

In [6]:
# ============================================================
# Comparaison PYTHON-SIDE : DMA AXI STREAM vs AXI FULL
# Objectif: comparer principalement les couts cote Python
# ============================================================
import numpy as np
import pandas as pd
from IPython.display import display

if "BENCH_SNAPSHOTS" not in globals():
    BENCH_SNAPSHOTS = {}

def _infer_label_from_context():
    bit = str(globals().get("BITFILE", "")).lower()
    if "sadf" in bit or "snr" in bit:
        return "DMA AXI STREAM"
    if "m_axi" in bit or "sam_axi" in bit:
        return "AXI FULL"
    return "UNKNOWN"

def _extract_current_df():
    if "df_all" in globals() and isinstance(df_all, pd.DataFrame) and len(df_all) > 0:
        return df_all.copy()
    if "records" in globals() and isinstance(records, list) and len(records) > 0:
        return pd.DataFrame(records)
    return None

def _meas_only(df):
    x = df.copy()
    if "warmup" in x.columns:
        x = x[~x["warmup"]]
    if "valid" in x.columns:
        x = x[x["valid"] == True]
    return x.copy()

def _style(df, title, float_cols):
    fmt = {c: "{:.6f}" for c in float_cols if c in df.columns}
    styled = (
        df.style
        .hide_index()
        .set_caption(title)
        .format(fmt)
        .set_table_styles([
            {"selector": "caption", "props": [
                ("caption-side", "top"),
                ("font-weight", "bold"),
                ("font-size", "14px"),
                ("text-align", "left"),
                ("padding", "8px 0"),
            ]},
            {"selector": "th", "props": [
                ("background-color", "#f2f2f2"),
                ("text-align", "center"),
                ("border", "1px solid #cccccc"),
                ("padding", "6px"),
            ]},
            {"selector": "td", "props": [
                ("text-align", "center"),
                ("border", "1px solid #cccccc"),
                ("padding", "6px"),
            ]},
            {"selector": "table", "props": [
                ("border-collapse", "collapse"),
                ("width", "100%"),
            ]},
        ])
    )
    display(styled)

def _safe_mean(df, col):
    if col not in df.columns or len(df) == 0:
        return np.nan
    return float(pd.to_numeric(df[col], errors="coerce").mean())

def _safe_p95(df, col):
    if col not in df.columns or len(df) == 0:
        return np.nan
    return float(pd.to_numeric(df[col], errors="coerce").quantile(0.95))

def _pick_snapshot(keys):
    for k in keys:
        if k in BENCH_SNAPSHOTS:
            return BENCH_SNAPSHOTS[k].copy(), k
    return None, None

# 1) Sauvegarde auto du contexte courant
current_df = _extract_current_df()
label = globals().get("CURRENT_BENCH_LABEL", _infer_label_from_context())
if current_df is not None:
    BENCH_SNAPSHOTS[label] = current_df.copy()
    print(f"[OK] Snapshot enregistre: '{label}' (rows={len(current_df)})")
else:
    print("[WARN] Aucune donnee benchmark detectee dans le contexte courant (df_all/records absents).")

print("[INFO] Snapshots disponibles:", list(BENCH_SNAPSHOTS.keys()))

# 2) Recuperation DMA / AXI FULL
df_dma, dma_name = _pick_snapshot(["DMA AXI STREAM", "DMA", "STREAM", "SA_df"])
df_axi, axi_name = _pick_snapshot(["AXI FULL", "M_AXI", "AXI", "mAXI"])

if df_dma is None or df_axi is None:
    print("\n[ACTION] Pour la comparaison cote Python:")
    print("1) Execute la cellule benchmark DMA")
    print("2) Execute la cellule benchmark AXI FULL")
    print("3) Relance cette cellule")
else:
    meas_dma = _meas_only(df_dma)
    meas_axi = _meas_only(df_axi)

    # Colonnes garanties
    for _df in [meas_dma, meas_axi]:
        for c in ["cache_ms", "cfg_ms", "ip_ms", "post_ms", "total_ms"]:
            if c not in _df.columns:
                _df[c] = np.nan
        _df["python_ms"] = pd.to_numeric(_df["cache_ms"], errors="coerce") + pd.to_numeric(_df["cfg_ms"], errors="coerce") + pd.to_numeric(_df["post_ms"], errors="coerce")
        _df["python_share_pct"] = np.where(pd.to_numeric(_df["total_ms"], errors="coerce") > 0, 100.0 * _df["python_ms"] / pd.to_numeric(_df["total_ms"], errors="coerce"), np.nan)
        _df["ip_share_pct"] = np.where(pd.to_numeric(_df["total_ms"], errors="coerce") > 0, 100.0 * pd.to_numeric(_df["ip_ms"], errors="coerce") / pd.to_numeric(_df["total_ms"], errors="coerce"), np.nan)

    # ============================================================
    # TABLEAU 1 - Comparison directe des temps cote Python
    # ============================================================
    table_py = pd.DataFrame([
        {
            "Mesure": "Prep cache (ms)",
            f"{dma_name} mean": _safe_mean(meas_dma, "cache_ms"),
            f"{axi_name} mean": _safe_mean(meas_axi, "cache_ms"),
            "Delta (AXI-DMA)": _safe_mean(meas_axi, "cache_ms") - _safe_mean(meas_dma, "cache_ms"),
            f"{dma_name} P95": _safe_p95(meas_dma, "cache_ms"),
            f"{axi_name} P95": _safe_p95(meas_axi, "cache_ms"),
        },
        {
            "Mesure": "Config registres (ms)",
            f"{dma_name} mean": _safe_mean(meas_dma, "cfg_ms"),
            f"{axi_name} mean": _safe_mean(meas_axi, "cfg_ms"),
            "Delta (AXI-DMA)": _safe_mean(meas_axi, "cfg_ms") - _safe_mean(meas_dma, "cfg_ms"),
            f"{dma_name} P95": _safe_p95(meas_dma, "cfg_ms"),
            f"{axi_name} P95": _safe_p95(meas_axi, "cfg_ms"),
        },
        {
            "Mesure": "Post-traitement (ms)",
            f"{dma_name} mean": _safe_mean(meas_dma, "post_ms"),
            f"{axi_name} mean": _safe_mean(meas_axi, "post_ms"),
            "Delta (AXI-DMA)": _safe_mean(meas_axi, "post_ms") - _safe_mean(meas_dma, "post_ms"),
            f"{dma_name} P95": _safe_p95(meas_dma, "post_ms"),
            f"{axi_name} P95": _safe_p95(meas_axi, "post_ms"),
        },
        {
            "Mesure": "Total Python (cache+cfg+post) (ms)",
            f"{dma_name} mean": _safe_mean(meas_dma, "python_ms"),
            f"{axi_name} mean": _safe_mean(meas_axi, "python_ms"),
            "Delta (AXI-DMA)": _safe_mean(meas_axi, "python_ms") - _safe_mean(meas_dma, "python_ms"),
            f"{dma_name} P95": _safe_p95(meas_dma, "python_ms"),
            f"{axi_name} P95": _safe_p95(meas_axi, "python_ms"),
        },
    ])
    _style(
        table_py,
        "COMPARAISON COTE PYTHON - DMA vs AXI FULL",
        [f"{dma_name} mean", f"{axi_name} mean", "Delta (AXI-DMA)", f"{dma_name} P95", f"{axi_name} P95"],
    )

    # ============================================================
    # TABLEAU 2 - Part Python vs part IP dans le total
    # ============================================================
    table_share = pd.DataFrame([
        {
            "Version": dma_name,
            "N runs": int(len(meas_dma)),
            "Mean total_ms": _safe_mean(meas_dma, "total_ms"),
            "Mean python_ms": _safe_mean(meas_dma, "python_ms"),
            "Mean ip_ms": _safe_mean(meas_dma, "ip_ms"),
            "Mean python_share_pct": _safe_mean(meas_dma, "python_share_pct"),
            "Mean ip_share_pct": _safe_mean(meas_dma, "ip_share_pct"),
            "P95 total_ms": _safe_p95(meas_dma, "total_ms"),
            "P95 python_ms": _safe_p95(meas_dma, "python_ms"),
            "P95 ip_ms": _safe_p95(meas_dma, "ip_ms"),
        },
        {
            "Version": axi_name,
            "N runs": int(len(meas_axi)),
            "Mean total_ms": _safe_mean(meas_axi, "total_ms"),
            "Mean python_ms": _safe_mean(meas_axi, "python_ms"),
            "Mean ip_ms": _safe_mean(meas_axi, "ip_ms"),
            "Mean python_share_pct": _safe_mean(meas_axi, "python_share_pct"),
            "Mean ip_share_pct": _safe_mean(meas_axi, "ip_share_pct"),
            "P95 total_ms": _safe_p95(meas_axi, "total_ms"),
            "P95 python_ms": _safe_p95(meas_axi, "python_ms"),
            "P95 ip_ms": _safe_p95(meas_axi, "ip_ms"),
        },
    ])
    _style(
        table_share,
        "REPARTITION TEMPORELLE - PART PYTHON VS PART IP",
        [
            "Mean total_ms", "Mean python_ms", "Mean ip_ms",
            "Mean python_share_pct", "Mean ip_share_pct",
            "P95 total_ms", "P95 python_ms", "P95 ip_ms",
        ],
    )

    # ============================================================
    # TABLEAU 3 - Debit effectif observé (informatif)
    # ============================================================
    table_bw = pd.DataFrame([
        {
            "Version": dma_name,
            "IN total mean (MB/s)": _safe_mean(meas_dma, "bw_in_total_MBps"),
            "OUT total mean (MB/s)": _safe_mean(meas_dma, "bw_out_total_MBps"),
            "IN IP mean (MB/s)": _safe_mean(meas_dma, "bw_in_ip_MBps"),
            "OUT IP mean (MB/s)": _safe_mean(meas_dma, "bw_out_ip_MBps"),
        },
        {
            "Version": axi_name,
            "IN total mean (MB/s)": _safe_mean(meas_axi, "bw_in_total_MBps"),
            "OUT total mean (MB/s)": _safe_mean(meas_axi, "bw_out_total_MBps"),
            "IN IP mean (MB/s)": _safe_mean(meas_axi, "bw_in_ip_MBps"),
            "OUT IP mean (MB/s)": _safe_mean(meas_axi, "bw_out_ip_MBps"),
        },
    ])
    _style(
        table_bw,
        "DEBITS OBSERVES - DMA vs AXI FULL",
        ["IN total mean (MB/s)", "OUT total mean (MB/s)", "IN IP mean (MB/s)", "OUT IP mean (MB/s)"],
    )

    print("\n[OK] Comparaison Python-side generee.")
    print("[INFO] Interprétation: table 1 compare strictement le cout Python, table 2 montre la part Python/IP dans le temps total.")

[OK] Snapshot enregistre: 'DMA AXI STREAM' (rows=12)
[INFO] Snapshots disponibles: ['DMA AXI STREAM', 'AXI FULL']


Mesure,DMA AXI STREAM mean,AXI FULL mean,Delta (AXI-DMA),DMA AXI STREAM P95,AXI FULL P95
Prep cache (ms),0.506327,0.506327,0.000000,0.517538,0.517538
Config registres (ms),0.259547,0.259547,0.000000,0.265624,0.265624
Post-traitement (ms),17.511203,17.511203,0.000000,17.581141,17.581141
Total Python (cache+cfg+post) (ms),18.277078,18.277078,0.000000,18.357485,18.357485


Version,N runs,Mean total_ms,Mean python_ms,Mean ip_ms,Mean python_share_pct,Mean ip_share_pct,P95 total_ms,P95 python_ms,P95 ip_ms
DMA AXI STREAM,10,2028.922444,18.277078,2010.611563,0.900827,99.097507,2029.476117,18.357485,2011.136659
AXI FULL,10,2028.922444,18.277078,2010.611563,0.900827,99.097507,2029.476117,18.357485,2011.136659


Version,IN total mean (MB/s),OUT total mean (MB/s),IN IP mean (MB/s),OUT IP mean (MB/s)
DMA AXI STREAM,0.047312,0.163364,0.047743,0.164851
AXI FULL,0.047312,0.163364,0.047743,0.164851



[OK] Comparaison Python-side generee.
[INFO] Interprétation: table 1 compare strictement le cout Python, table 2 montre la part Python/IP dans le temps total.


In [14]:
from pathlib import Path

# Dossier d'export (adapter si besoin)
export_dir = Path("/home/xilinx/jupyter_notebooks/Ibrahim/algPython/axi_full_vs_dma/comparaison_dma_axi_full")
export_dir.mkdir(parents=True, exist_ok=True)

# Vérification des tables
if "table_py" not in globals() or "table_share" not in globals() or "table_bw" not in globals():
    raise RuntimeError("Exécute d'abord la cellule de comparaison pour créer table_py, table_share et table_bw.")

# Export direct des DataFrames déjà formatés
table_py.to_csv(export_dir / "comparaison_cote_python.csv", index=False)
table_share.to_csv(export_dir / "repartition_temporelle.csv", index=False)
table_bw.to_csv(export_dir / "debits_observes.csv", index=False)

print("Exports OK")
print((export_dir / "comparaison_cote_python.csv").resolve())
print((export_dir / "repartition_temporelle.csv").resolve())
print((export_dir / "debits_observes.csv").resolve())

Exports OK
/home/xilinx/jupyter_notebooks/Ibrahim/algPython/axi_full_vs_dma/comparaison_dma_axi_full/comparaison_cote_python.csv
/home/xilinx/jupyter_notebooks/Ibrahim/algPython/axi_full_vs_dma/comparaison_dma_axi_full/repartition_temporelle.csv
/home/xilinx/jupyter_notebooks/Ibrahim/algPython/axi_full_vs_dma/comparaison_dma_axi_full/debits_observes.csv
